# 此代码使用时需将GPU设置为：GPU T4*2


In [ ]:
import time

NOTEBOOK_START_TIME = time.time()

In [ ]:
DEBUG = False

QUANT = False

if QUANT:
    from transformers import BitsAndBytesConfig
    quantization_config = BitsAndBytesConfig(
        load_in_4bit = True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

USE_PAST_KEY = True

In [ ]:
QUANT

In [ ]:
%%time
if QUANT:
    !pip install -U /kaggle/input/accelerate-wheelwhl/accelerate-0.29.1-py3-none-any.whl -qq
    !pip install -U /kaggle/input/bitsandbytes-0-42-0-py3-none-any-whl/bitsandbytes-0.42.0-py3-none-any.whl -qq


import torch
import gc
torch.backends.cuda.enable_mem_efficient_sdp(False)

from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    AutoConfig,
    StoppingCriteria,
    set_seed
)

import transformers
print(f"Transformers Version: {transformers.__version__}")
set_seed(42)

In [ ]:
import pandas as pd
from tqdm import tqdm
PRIVATE = True

df = pd.read_csv('/kaggle/input/ai-mathematical-olympiad-prize/test.csv')
df.head()

In [ ]:
if len(df) < 5:
    df = pd.read_csv('/kaggle/input/ai-mathematical-olympiad-prize/train.csv')
    PRIVATE = False
df.head()

In [ ]:
df.shape

In [ ]:
def naive_parse(answer):
    out = []
    start = False
    end = False
    for l in reversed(list(answer)):
        if l in '0123456789' and not end:
            start = True
            out.append(l)
        else:
            if start:
                end = True
        
    out = reversed(out)
    return ''.join(out)

In [ ]:
import re  # 导入正则表达式模块
import sys  # 导入系统模块，用于访问系统特性和操作
import subprocess  # 导入子进程模块，用于执行外部命令

# 返回输出中最后打印的内容
def return_last_print(output, n):
    lines = output.strip().split('\n')  # 去除输出两边的空格并按行分割
    if lines:
        return lines[n]  # 返回第n行内容
    else:
        return ""  # 如果没有内容，则返回空字符串

# 处理代码的函数
def process_code(code, return_shell_output=False):
    
    # 替换符合正则表达式的内容
    def repl(match):
        if "real" not in match.group():
            return "{}{}".format(match.group()[:-1], ', real=True)')  # 如果匹配到的内容不包含“real”，则添加real=True
        else:
            return "{}{}".format(match.group()[:-1], ')')  # 如果包含“real”，则仅在末尾添加括号
    
    code = re.sub(r"symbols\([^)]+\)", repl, code)  # 使用正则替换符号部分

    # 如果需要返回shell输出，则将代码修改为符合shell执行的格式
    if return_shell_output:
        code = code.replace('\n', '\n    ')  # 将每行代码缩进，以便能嵌套到try块中
        # 添加异常处理代码，确保执行时出错能够捕获并打印错误
        code = "\ntry:\n    from sympy import *\n{}\nexcept Exception as e:\n    print(e)\n    print('FAIL')\n".format(code)
    
    # 如果不返回shell输出，则直接打印代码
    if not return_shell_output:
        print(code)
    
    # 将代码写入文件 'code.py'
    with open('code.py', 'w') as fout:
        fout.write(code)
    
    # 通过子进程执行生成的 Python 代码
    batcmd = 'timeout 7 ' + sys.executable + ' code.py'  # 设置命令，运行超时时间为7秒
    try:
        # 使用 subprocess 执行命令并获取输出
        shell_output = subprocess.check_output(batcmd, shell=True).decode('utf8')
        # 获取输出中的最后一行并打印
        return_value = return_last_print(shell_output, -1)
        print(shell_output)
        
        # 如果需要返回shell输出，则判断执行结果是否成功
        if return_shell_output:
            if return_value == 'FAIL':  # 如果返回“FAIL”，表示执行失败
                CODE_STATUS = False
                return_value = return_last_print(shell_output, -2)  # 获取倒数第二行的错误信息
                if "not defined" in return_value:  # 如果是未定义的错误，则提示检查格式和导入
                    return_value += '\nTry checking the formatting and imports'
            else:
                CODE_STATUS = True  # 如果没有失败，则表示成功
            return return_value, CODE_STATUS  # 返回结果和状态
        
        # 如果返回值是数字，则将其转化为整数并取余数
        code_output = round(float(eval(return_value))) % 1000
    except Exception as e:
        print(e, 'shell_output')  # 捕获异常并打印
        code_output = -1  # 如果执行失败，返回-1
    
    # 如果需要返回shell输出，则判断执行结果并返回
    if return_shell_output:
        if code_output == -1:
            CODE_STATUS = False
        else:
            CODE_STATUS = True
        return code_output, CODE_STATUS  
    
    return code_output  # 返回结果

# 处理文本输出，提取数字结果
def process_text_output(output):
    result = output  # 将输出赋值给result
    try:
        # 使用正则表达式查找输出中符合\boxed{数字}的内容
        result_output = re.findall(r'\\boxed\{(\d+)\}', result)

        print('BOXED', result_output)
        if not len(result_output):  # 如果没有找到\boxed，则调用naive_parse进行处理
            result_output = naive_parse(result)
        else:
            result_output = result_output[-1]  # 如果找到多个结果，则取最后一个

        print('BOXED FINAL', result_output)
        if not len(result_output):  # 如果仍然没有结果，则返回-1
            result_output = -1
        
        else:
            result_output = round(float(eval(result_output))) % 1000  # 将结果转化为数字并取余数
    
    except Exception as e:
        print(e)  # 捕获异常并打印
        print('ERROR PARSING TEXT')

In [ ]:
torch.cuda.empty_cache()
gc.collect()

# 将private设置为true，以方便在本地使用deepseek进行数学题推理

In [ ]:
PRIVATE = True

In [ ]:
import re
import math
import random

from collections import defaultdict

n_repetitions = 22 if PRIVATE else 4
TOTAL_TOKENS = 2048 # if PRIVATE else 512

if PRIVATE:
    TIME_LIMIT = 31500
else:
    TIME_LIMIT = 1

In [ ]:
print(f'n_repetitions;{n_repetitions}')
print(f'TOTAL_TOKENS:{TOTAL_TOKENS}')
print(f'PRIVATE:{PRIVATE}')
print(f'TIME_LIMIT:{TIME_LIMIT}')
print(f'QUANT:{QUANT}')

In [ ]:
import torch
print(torch.cuda.device_count())  # 输出可用的GPU数量


In [ ]:
QUANT

In [ ]:
if PRIVATE:

    MODEL_PATH = '/kaggle/input/deepseek-math'#"/root/autodl-tmp/AIMO1/20th/model_data/deepseek-math-7b-rl"#"/kaggle/input/gemma/transformers/7b-it/1"
    DEEP = True

    config = AutoConfig.from_pretrained(MODEL_PATH)
    config.gradient_checkpointing = True

    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

    device_map = [('model.embed_tokens', 0),
                 ('model.layers.0', 0),
                 ('model.layers.1', 0),
                 ('model.layers.2', 0),
                 ('model.layers.3', 0),
                 ('model.layers.4', 0),
                 ('model.layers.5', 0),
                 ('model.layers.6', 0),
                 ('model.layers.7', 0),
                 ('model.layers.8', 0),
                 ('model.layers.9', 0),
                 ('model.layers.10', 0),
                 ('model.layers.11', 0),
                 ('model.layers.12', 0),
                 ('model.layers.13', 0),
                 ('model.layers.14', 0),
                 ('model.layers.15', 0),
                 ('model.layers.16', 0),
                 ('model.layers.17', 0),
                 ('model.layers.18', 0),
                 ('model.layers.19', 0),
                 ('model.layers.20', 0),
                 ('model.layers.21', 0),
                 ('model.layers.22', 1),
                 ('model.layers.23', 1),
                 ('model.layers.24', 1),
                 ('model.layers.25', 1),
                 ('model.layers.26', 1),
                 ('model.layers.27', 1),
                 ('model.layers.28', 1),
                 ('model.layers.29', 1),
                 ('model.norm', 1),
                 ('lm_head', 1)]

    device_map = {ii:jj for (ii,jj) in device_map}

    if QUANT:
        from transformers import BitsAndBytesConfig
        quantization_config = BitsAndBytesConfig(
            load_in_4bit = True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            device_map="sequential",
            torch_dtype="auto",
            trust_remote_code=True, 
            quantization_config=quantization_config,
            config=config
        )
    else: # 此处quant==False运行这一部分 
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            device_map=device_map,
            torch_dtype="auto",
            trust_remote_code=True,
            #quantization_config=quantization_config,
            config=config
        )
    # else: # 此处quant==False运行这一部分 
    #     model = AutoModelForCausalLM.from_pretrained(
    #         MODEL_PATH,
    #         device_map='auto',
    #         torch_dtype="auto",
    #         trust_remote_code=True,
    #         #quantization_config=quantization_config,
    #         config=config
    #     )
    pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype='auto',
    device_map=device_map,
)
    from transformers import StoppingCriteriaList

    class StoppingCriteriaSub(StoppingCriteria):
        def __init__(self, stops = [], encounters=1):
            super().__init__()
            self.stops = [stop.to("cuda") for stop in stops]

        def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor):
            for stop in self.stops:
                last_token = input_ids[0][-len(stop):]
                if torch.all(torch.eq(stop,last_token)):
                    return True
            return False


    stop_words = ["```output", "```python", "```\nOutput" , ")\n```" , "``````output"] #,  
    stop_words_ids = [tokenizer(stop_word, return_tensors='pt', add_special_tokens=False)['input_ids'].squeeze() for stop_word in stop_words]
    stopping_criteria = StoppingCriteriaList([StoppingCriteriaSub(stops=stop_words_ids)])
    
    model.dtype, model.hf_device_map

In [ ]:
code = """Below is a math problem you are to solve (positive numerical answer):
\"{}\"
To accomplish this, first determine a sympy-based approach for solving the problem by listing each step to take and what functions need to be called in each step. Be clear so even an idiot can follow your instructions, and remember, your final answer should be positive integer, not an algebraic expression!
Write the entire script covering all the steps (use comments and document it well) and print the result. After solving the problem, output the final numerical answer within \\boxed{}.

Approach:"""


cot = """Below is a math problem you are to solve (positive numerical answer!):
\"{}\"
Analyze this problem and think step by step to come to a solution with programs. After solving the problem, output the final numerical answer within \\boxed{}.\n\n"""

promplt_options = [code,cot]

In [ ]:
promplt_options

In [ ]:
import re
from collections import defaultdict
from collections import Counter

from numpy.random import choice
import numpy as np

tool_instruction = '\n\nPlease integrate natural language reasoning with programs to solve the above problem, and put your final numerical answer within \\boxed{}.\nNote that the intermediary calculations may be real numbers, but the final numercal answer would always be an integer.'


#tool_instruction = " The answer should be given as a non-negative modulo 1000."
#tool_instruction += '\nPlease integrate natural language reasoning with programs to solve the problem above, and put your final answer within \\boxed{}.'

temperature = 0.9
top_p = 1.0

temperature_coding = 0.9
top_p_coding = 1.0
   
total_results = {}
total_answers = {}
best_stats = {}
total_outputs = {}
question_type_counts = {}
starting_counts = (2,3)

In [ ]:
n_repetitions = 1
df = df.head(1)
df

# debug


In [ ]:
for jj in tqdm(range(n_repetitions)):   
    for i in tqdm(range(len(df))):
        TIME_SPENT = time.time() - NOTEBOOK_START_TIME
        if TIME_SPENT>TIME_LIMIT:
            break
        id_ = df['id'].loc[i]
        problem = df['problem'].loc[i]
        print('#'*10)
        print(f"**\n\n\nQUESTION 第{jj}轮，第{i}个问题 - 目前为止耗时 : {TIME_SPENT:.0f} secs")
        
        best, best_count = best_stats.get(i,(-1,-1))
        # print(f'best:{best}')
        # print(f'best_count:{best_count}')
        
        if best_count>np.sqrt(jj):
            print("**SKIPPING CAUSE ALREADY FOUND BEST")
            continue
        outputs = total_outputs.get(i,[])
        # print(f'outputs:{outputs}')
        # 最开始给填充的默认值
        text_answers, code_answers = question_type_counts.get(i,starting_counts)#starting_counts是没有找到时返回的默认值
        print(f'text_answers:{text_answers}')
        print(f'code_answers:{code_answers}')
        
        results = total_results.get(i,[])
        answers = total_answers.get(i,[])

        # print(f'results:{results}')
        # print(f'answers:{answers}')
        
        print('**使用循环五次进行显存清除')
        for _ in range(5):
            torch.cuda.empty_cache()
            gc.collect()
            time.sleep(0.2)
        #使用模型进行生成
        print('###################################下面进行问题的预准备部分#################################')
        
        try:#
            print('¥¥¥¥¥¥¥¥¥¥¥¥¥此处进入try¥¥¥¥¥¥¥¥¥¥¥¥¥¥')
            ALREADY_GEN = 0
            code_error = None
            code_error_count = 0
            code_output = -1
            #initail_message = problem  + tool_instruction 
            counts = np.array([text_answers,code_answers])
            print(f'**counts:       {counts}')
            #==========================================
            draw = choice(promplt_options, 1,
                          p=counts/counts.sum())
            #draw是选择提示词使用，从promplt_options中选择，选择1个元素，p是指示每个被选中的概率
            # print(f'选取的提示词模板:         {draw}')
            # 将问题通过format语法填充到提示词中
            initail_message = draw[0].format(problem,"{}")   
            
            print(f'############################1——将具体问题填充到提示词模板后的内容########################:')
            prompt = f"User: {initail_message}"
            current_printed = len(prompt)
            # print(f'添加了User后文本的长度(未编码)：{current_printed}')
            print(f"{jj}_{prompt}\n")
            model_inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
            input_len = len(model_inputs['input_ids'][0])
            print(f'**文本编码后input_ids的长度：           {input_len}')
            print('###################################2——下面进行model产生回答的部分#################################')
            generation_output = model.generate(**model_inputs, 
                                               max_new_tokens=TOTAL_TOKENS-ALREADY_GEN,
                                               return_dict_in_generate=USE_PAST_KEY,
                                               do_sample = True,
                                               temperature = temperature,
                                               top_p = top_p,
                                               num_return_sequences=1, stopping_criteria = stopping_criteria)
            print()
            if USE_PAST_KEY:
                output_ids = generation_output.sequences[0]
            else:
                output_ids = generation_output[0]
            # print(f'检查模型生成返回的内容的长度{output_ids.shape}')
            
            decoded_output = tokenizer.decode(output_ids, skip_special_tokens=True)
            print('**大模型生成的回答解码后的内容：')
            print(decoded_output)
            print('**解码后裁剪出来的内容(删除原始的问题，只保留了新生成的回答)')
            print(f"{decoded_output[current_printed:]}\n")
            current_printed += len(decoded_output[current_printed:])
            
            print(f'**打印目前生成的回答:{current_printed}')
            cummulative_code = ""
            
            stop_word_cond = False
            print('**检测生成的文本最后是否有终止词')
            print(f'**生成的文本的最后部分：{decoded_output[-10:]}')
            
            for stop_word in stop_words:
                stop_word_cond = stop_word_cond or (decoded_output[-len(stop_word):]==stop_word)
            print(f'**是否满足终止词条件:{stop_word_cond}')  
            while (stop_word_cond) and (ALREADY_GEN<(TOTAL_TOKENS)):
                print('###########3--当生成的内容符合要求（句子完整）并且生成的长度还没有超过规定的最大长度时############')
                
                
                if (decoded_output[-len("```python"):]=="```python"):
                    temperature_inner=temperature_coding
                    top_p_inner = top_p_coding
                    prompt = decoded_output
                else:
                    temperature_inner=temperature
                    top_p_inner = top_p
                    try:
                        if (decoded_output[-len("``````output"):]=="``````output"):
                            code_text = decoded_output.split('```python')[-1].split("``````")[0]
                        else:
                            code_text = decoded_output.split('```python')[-1].split("```")[0]
                        
                        print('**代码累计开始的标志点')
                        print(f'**打印现在已经累计的代码：{cummulative_code}')
                        print(f'**新生成的代码：{code_text}')
                        
                        cummulative_code+=code_text
                        #########在这里将生成的代码写入到code文件中，然后运行代码查看结果z
                        code_output, CODE_STATUS = process_code(cummulative_code, return_shell_output=True)
                        print('**CODE RESULTS', code_output)
                        print('**CODE_STATUS', CODE_STATUS)
                        #TOTAL_TOKENS:True说明程序运行正常，否则说明生成的程序有问题
                        if code_error==code_output:# 如果程序输出为Nane，判断程序出错p
                            
                            code_error_count+=1
                        else:
                            code_error=code_output
                            code_error_count = 0

                        if not CODE_STATUS:#如果程序出错了，我把生成的程序删除，整理prompt后面用r
                            cummulative_code = cummulative_code[:-len(code_text)]#相当于裁剪掉了生成的代码部分x
                            #若果发现持续出现错误，就停止循环r
                            if code_error_count>=1:
                                print("REPEATED ERRORS")
                                break

                    except Exception as e:
                        print(e)
                        print('ERROR PARSING CODE')
                        code_output = -1

                    if code_output!=-1:
                        if (decoded_output[-len(")\n```"):]==")\n```"):
                            prompt = decoded_output+'```output\n'+str(code_output)+'\n```\n'
                        else:
                            prompt = decoded_output+'\n'+str(code_output)+'\n```\n'
                    else:
                        prompt = decoded_output
                        cummulative_code=""

                
                print('%'*200)
                print('此处跳出循环')
                print('########################4--此处开始生成第二次调用大模型的代码########################')
                
                print('**prompt:',prompt)
                model_inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
                ALREADY_GEN =  len(model_inputs['input_ids'][0])-input_len

                if USE_PAST_KEY:
                    old_values = generation_output.past_key_values
                else:
                    old_values = None

                generation_output = model.generate(**model_inputs, 
                                                   max_new_tokens=TOTAL_TOKENS-ALREADY_GEN, 
                                                   return_dict_in_generate=USE_PAST_KEY,
                                                   past_key_values=old_values,
                                                   do_sample = True,
                                                   temperature = temperature_inner,
                                                   top_p = top_p_inner,
                                                   num_return_sequences=1, stopping_criteria = stopping_criteria)

                if USE_PAST_KEY:
                    output_ids = generation_output.sequences[0]
                else:
                    output_ids = generation_output[0]
                decoded_output = tokenizer.decode(output_ids, skip_special_tokens=True)

                print('**这里根据指导的思路打印生成的代码')
                print(f"**\nINTERMEDIATE OUT :\n{decoded_output[current_printed:]}\n")
                current_printed+=len(decoded_output[current_printed:])
                
                stop_word_cond = False
                for stop_word in stop_words:
                    stop_word_cond = stop_word_cond or (decoded_output[-len(stop_word):]==stop_word)

                #While的终止条件通过这里检查新的stop_word_cond来生成l
                print()
            ####################################################### 此处为while循环的截止位置   #######################################################



            
            print()
            print()
            print()
            
            print('**此处已经跳出代码的生成循环，不再使用大模型生成python代码，做一些后处理相关内容')
            if USE_PAST_KEY:
                output_ids = generation_output.sequences[0]
            else:
                output_ids = generation_output[0]

            raw_output = tokenizer.decode(output_ids[input_len:], skip_special_tokens=True)
            #print(f"\n\nOutput :\n{raw_output}\n")                            
            result_output = process_text_output(raw_output)
            
            try:
                code_output = round(float(eval(code_output))) % 1000
            except Exception as e:
                print(e,'final_eval')
                code_output = -1

        except Exception as e:
            print(e,"5")
            result_output, code_output = -1, -1

        if code_output!=-1:
            outputs.append(code_output)
            code_answers+=1

        if result_output!=-1:
            outputs.append(result_output)
            text_answers+=1

        if len(outputs) > 0:
            occurances = Counter(outputs).most_common()
            print(occurances)
            if occurances[0][1] > best_count:
                print("GOOD ANSWER UPDATED!")
                best = occurances[0][0]
                best_count = occurances[0][1]
            if occurances[0][1] > 5:
                print("ANSWER FOUND!")
                break

        results.append(result_output)
        answers.append(code_output)
        
        best_stats[i] = (best, best_count) 
        question_type_counts[i] = (text_answers, code_answers)
        total_outputs[i] = outputs
        
        total_results[i] = results
        total_answers[i] = answers

        print("code_answers",code_answers-starting_counts[1],"text_answers",text_answers-starting_counts[0])
        if DEBUG:
            break
            
        print('代码运行完成')

In [ ]:
1

In [ ]:
import numpy as np
if PRIVATE:
    for ii in range(len(df)):
        a = total_answers[ii]
        b = total_answers[ii]
        a = np.array(a)
        b = np.array(b)
        print(a,b)
        a[a < 0] = b[a < 0]

        pred = Counter(a.tolist()).most_common(2)
        print(pred)

In [ ]:
if PRIVATE:
    df['answer'] = [best_stats[ii][0] for ii in range(len(df))]
else:
    df['answer'] = 2

In [ ]:
df.head()

In [ ]:
if not PRIVATE:
    print(1)
    df = pd.read_csv('/kaggle/input/ai-mathematical-olympiad-prize/train.csv')
    if PRIVATE:
        df['model_answer'] = [best_stats[ii][0] for ii in range(len(df))]
        df['match'] = df.answer == df.model_answer
        print(f'{df.match.sum()} matches in {len(df)} examples')

In [ ]:
df

In [ ]:
with open('code.py', 'w') as fout:
    fout.write("print('done')")

batcmd = 'timeout 7 ' + sys.executable + ' code.py'
try:
    shell_output = subprocess.check_output(batcmd, shell=True).decode('utf8')
    print(shell_output)
except:
    pass

# 下面为测试的代码x

# 将字符串版本的代码转化为可执行代码的测试z

In [ ]:
OK_CODE = '''
from sympy import symbols, Eq, solve, sqrt, simplify

def distance_squared_sum():
    x = symbols('x')
    k, l = symbols('k l', positive=True, real=True)

    # Set the equations equal to each other
    equation = Eq(k*x**2 - 2*k*x + l, 4)
    
    # Solve for x
    solutions = solve(equation, x)
    A, B = solutions

    # Calculate the distances from A and B to the origin
    dA = sqrt(A**2 + (k*A**2 - 2*k*A + l)**2)
    dB = sqrt(B**2 + (k*B**2 - 2*k*B + l)**2)

    # Simplify the expressions
    dA = simplify(dA)
    dB = simplify(dB)

    # Calculate the sum of the squares of the distances
    sum_of_squares = dA**2 + dB**2

    return sum_of_squares.subs({k: 1, l: 2})

result = distance_squared_sum()
print(result)
'''


In [ ]:
# 处理代码的函数
def process_code_DeBUG(code, return_shell_output=False):
    
    # 替换符合正则表达式的内容
    print(f'repl处理之前的Code：{code}')
    def repl(match):
        if "real" not in match.group():
            return "{}{}".format(match.group()[:-1], ', real=True)')  # 如果匹配到的内容不包含“real”，则添加real=True
        else:
            return "{}{}".format(match.group()[:-1], ')')  # 如果包含“real”，则仅在末尾添加括号
    
    code = re.sub(r"symbols\([^)]+\)", repl, code)  # 使用正则替换符号部分
    print(f'处理后Code：{code}')

    # 如果需要返回shell输出，则将代码修改为符合shell执行的格式
    if return_shell_output:
        code = code.replace('\n', '\n    ')  # 将每行代码缩进，以便能嵌套到try块中
        # 添加异常处理代码，确保执行时出错能够捕获并打印错误
        code = "\ntry:\n    from sympy import *\n{}\nexcept Exception as e:\n    print(e)\n    print('FAIL')\n".format(code)
    
    # 如果不返回shell输出，则直接打印代码
    if not return_shell_output:
        print(code)
    
    # 将代码写入文件 'code.py'
    with open('code.py', 'w') as fout:
        fout.write(code)
    
    # 通过子进程执行生成的 Python 代码
    batcmd = 'timeout 7 ' + sys.executable + ' code.py'  # 设置命令，运行超时时间为7秒
    try:
        # 使用 subprocess 执行命令并获取输出
        shell_output = subprocess.check_output(batcmd, shell=True).decode('utf8')
        # 获取输出中的最后一行并打印
        return_value = return_last_print(shell_output, -1)
        print(shell_output)
        
        # 如果需要返回shell输出，则判断执行结果是否成功
        if return_shell_output:
            if return_value == 'FAIL':  # 如果返回“FAIL”，表示执行失败
                CODE_STATUS = False
                return_value = return_last_print(shell_output, -2)  # 获取倒数第二行的错误信息
                if "not defined" in return_value:  # 如果是未定义的错误，则提示检查格式和导入
                    return_value += '\nTry checking the formatting and imports'
            else:
                CODE_STATUS = True  # 如果没有失败，则表示成功
            return return_value, CODE_STATUS  # 返回结果和状态
        
        # 如果返回值是数字，则将其转化为整数并取余数
        code_output = round(float(eval(return_value))) % 1000
    except Exception as e:
        print(e, 'shell_output')  # 捕获异常并打印
        code_output = -1  # 如果执行失败，返回-1
    
    # 如果需要返回shell输出，则判断执行结果并返回
    if return_shell_output:
        if code_output == -1:
            CODE_STATUS = False
        else:
            CODE_STATUS = True
        return code_output, CODE_STATUS  
    
    return code_output  # 返回结果


In [ ]:
code_output_11, CODE_STATUS_11 = process_code_DeBUG(OK_CODE, return_shell_output=True)
#code_output:程序运行的答案
# CODE_STATUS：程序能否正常运行

In [ ]:
code_output_11

In [ ]:
CODE_STATUS_11